# CMA-ES Optimization for Worst-Case OMWU Dynamics
This notebook demonstrates how to use the `CMAESGameOptimizer` to find payoff matrices that cause the Optimistic Multiplicative Weights Update (OMWU) algorithm to suffer maximum asymptotic regret in a 2-player, 2-action game.


In [ ]:
import sys
sys.path.append("..")

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.config.schemas import ExperimentConfig, GameConfig, DynamicConfig, ExecutionConfig
from src.engine.optimizer import CMAESGameOptimizer
from src.engine.runner import ExperimentRunner
from src.engine.statistics import load_experiment_stats

# Set formatting for plots
plt.style.use('ggplot')


## 1. Configure the Optimizer
We configure a 2x2 game base template and set the CMA-ES optimizer to target the `delta_reg` objective. This helps us find games where the algorithm continues to diverge or cycle heavily even late into the simulation.


In [ ]:
# Setup a small configuration for 2x2 games
# We use a batch size of 20 to run 20 games in parallel per generation
population_size = 20

config = ExperimentConfig(
    name="cmaes_omwu_demo",
    game=GameConfig(
        generator="custom",
        utility_range=(-1.0, 1.0),
        payoffs=[
            [[0.0, 0.0], [0.0, 0.0]], # Player 1 base
            [[0.0, 0.0], [0.0, 0.0]], # Player 2 base
        ]
    ),
    dynamic=DynamicConfig(
        algorithm="omwu", 
        eta=0.1
    ),
    execution=ExecutionConfig(
        total_steps=1000, 
        batch_size=population_size,
        steps_per_call=200, 
        device="auto",
        compile=True
    )
)

# Initialize CMA-ES Optimizer
# T1_ratio = 0.5 means we compute delta regret from step 500 to 1000
optimizer = CMAESGameOptimizer(
    base_config=config, 
    sigma=0.5, 
    seed=42,
    objective_type="delta_reg",
    T1_ratio=0.5,
    lambda_reg=0.1
)


## 2. Run the Optimization
We run the optimizer for a few generations (e.g., 5). In a real experiment, you might run this for 50-100 generations.


In [ ]:
# Run optimization
best_payoffs, best_regret = optimizer.optimize(generations=5)

print("Optimization Complete!")
print(f"Worst-case Regret Found: {best_regret:.4f}")
print("\nWorst-case Payoff Matrix Player 1:")
print(np.round(best_payoffs[0].numpy(), 4))
print("\nWorst-case Payoff Matrix Player 2:")
print(np.round(best_payoffs[1].numpy(), 4))


## 3. Simulate the Worst-Case Game
Now we plug these worst-case matrices back into a standard `ExperimentRunner` and run a longer, high-fidelity simulation (e.g. 5000 steps) to see what the dynamics look like.


In [ ]:
import copy

# Create a validation config with the worst-case matrices
val_config = copy.deepcopy(config)
val_config.game.payoffs = [p.numpy().tolist() for p in best_payoffs]
val_config.execution.batch_size = 1
val_config.execution.total_steps = 5000
val_config.execution.steps_per_call = 500
val_config.name = "worst_case_omwu"

# Run the single simulation
runner = ExperimentRunner(val_config)
summary = runner.run()

print(f"Validation Run Complete. Session ID: {summary['session_id']}")


## 4. Plot the Trajectories
We extract the cumulative regret and strategy probabilities using `load_experiment_stats` and visualize the non-converging dynamics.


In [ ]:
# Load the recorded statistics from disk
stats_data = load_experiment_stats(output_dir=summary["output_dir"], session_id=summary["session_id"])
steps = stats_data["steps"].numpy()
cum_regrets = stats_data["cum_regrets"].numpy()
strats = stats_data["strategies"]  # List of 2D tensors [T x A_i] per player

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Regret Curve
axes[0].plot(steps, cum_regrets[:, 0], label="Player 1 Cum Regret", color="crimson")
axes[0].plot(steps, cum_regrets[:, 1], label="Player 2 Cum Regret", color="navy", linestyle="--")
axes[0].set_xlabel("Step T")
axes[0].set_ylabel("Cumulative Regret")
axes[0].set_title("OMWU Cumulative Regret Trajectory")
axes[0].grid(True)
axes[0].legend()

# Strategy Trajectory for Player 1 & 2 (Probability of Action 0)
axes[1].plot(steps, strats[0][:, 0].numpy(), color="purple", label="Player 1 Pr(Action 0)")
axes[1].plot(steps, strats[1][:, 0].numpy(), color="orange", label="Player 2 Pr(Action 0)", linestyle="--")
axes[1].set_xlabel("Step T")
axes[1].set_ylabel("Probability")
axes[1].set_title("Strategy Evolution on Simplex")
axes[1].grid(True)
axes[1].legend()

plt.tight_layout()
plt.show()
